# 02 — Partições e anotações

Objetivo: verificar grupos separados e schema de referência antes dos recortes. Entradas: manifesto e anotações configuráveis. Saídas: validação e template pendente. Pré-requisitos: notebook 01 ou manifesto existente. A divisão sintética 1/1/2 serve apenas ao teste; não é recomendação para dados reais.

Execução: na raiz, `python -m jupyter lab notebooks`. Execute de cima para baixo. Variáveis: `TCC_ROOT`, `TCC_RUN_MODE=synthetic|real`, `TCC_MANIFEST`, `TCC_ANNOTATIONS`, `TCC_SPLIT`. Colab pode definir `TCC_ROOT` para a cópia do repositório e instalar as dependências em runtime Python compatível.

In [ ]:
import os, sys, json, subprocess
from pathlib import Path
ROOT = Path(os.environ.get("TCC_ROOT", Path.cwd())).resolve()
if not (ROOT / "src").is_dir() and (ROOT.parent / "src").is_dir():
    ROOT = ROOT.parent
assert (ROOT / "src").is_dir(), "Defina TCC_ROOT para a raiz do repositório."
sys.path.insert(0, str(ROOT))
MODE = os.environ.get("TCC_RUN_MODE", "synthetic")
assert MODE in {"synthetic", "real"}
from scripts.notebook_support import prepare_synthetic, SyntheticClassifier
from src import data_protocol as dp
if MODE == "synthetic":
    OUTPUT, rows = prepare_synthetic(ROOT)
    MANIFEST = OUTPUT / "images.csv"
    ANNOTATIONS = OUTPUT / "synthetic_annotations.json"
else:
    OUTPUT = ROOT / "experiments" / "runs" / "notebook_real"
    OUTPUT.mkdir(parents=True, exist_ok=True)
    MANIFEST = Path(os.environ.get("TCC_MANIFEST", ROOT / "data/splits/trypomastigote/images.csv"))
    ANNOTATIONS = Path(os.environ.get("TCC_ANNOTATIONS", ROOT / "data/annotations/experimento_reference_v2.json"))
if MODE == "real" and not MANIFEST.is_file():
    raise FileNotFoundError("Partições reais ainda não preparadas. Siga docs/guia_treinamento.md, seção 4, e defina TCC_MANIFEST e TCC_ANNOTATIONS.")
print({"modo": MODE, "raiz": str(ROOT), "manifesto": str(MANIFEST)})


In [ ]:
rows = dp.read_manifest(MANIFEST)
dp.validate_partitions(rows)
annotations = dp.load_annotations(ANNOTATIONS, rows)
print({"imagens":len(rows), "grupos":len({r["group_id"] for r in rows}),
       "revisadas":sum(a["review_status"]=="reviewed" for a in annotations.values())})
if MODE == "synthetic":
    pending = json.loads((OUTPUT / "pending.json").read_text())
    assert all(a["whole_image_label"] is None for a in pending["images"])
    print("Referência supervisionada é fixture sintética; template real continua sem rótulos.")


Em dados reais, use `scripts/prepare_data.py split --assignments grupos.csv` após examinar os grupos/classes. Rotulagem exige equipe qualificada. Execute `notebooks/gerar_rois.py --image-manifest ... --annotations ... --config configs/model_b.yaml --output-dir ...` somente com manifesto persistido. Não anotado continua desconhecido.